# บทที่ 10 - ตัวแทน AI ในการผลิต

ในบทนี้คุณจะได้เรียนรู้ **รูปแบบการผลิต** สำหรับตัวแทน AI โดยใช้ Microsoft Agent Framework (Python) เราจะครอบคลุม:

- **การสังเกตการณ์** — การเพิ่มการจับเวลาและการบันทึกการโต้ตอบของตัวแทน
- **การประเมินผล** — การใช้ตัวแทนผู้ประเมินเพื่อให้คะแนนคุณภาพการตอบสนอง
- **การจัดการค่าใช้จ่าย** — กลยุทธ์สำหรับการเพิ่มประสิทธิภาพโทเค็นและการเลือกโมเดล

บทบาทสมมติคือ **ตัวแทนท่องเที่ยว** ที่ช่วยผู้ใช้วางแผนการเดินทาง พร้อมด้วยการติดตามและประเมินผลในชั้นบน


## การตั้งค่า


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## การพิจารณาการใช้งานในสภาพแวดล้อมจริง

การย้ายตัวแทน AI จากต้นแบบไปสู่การผลิตต้องให้ความสำคัญอย่างรอบคอบกับสามเสาหลัก:

1. **การสังเกตการณ์** — คุณจำเป็นต้องมีการมองเห็นว่าตัวแทนกำลังทำอะไร ใช้เวลานานเท่าไร และเรียกใช้เครื่องมือใดบ้าง หากไม่มีการติดตามและบันทึกเหตุการณ์ การแก้ไขข้อผิดพลาดในสภาพแวดล้อมจริงแทบจะเป็นไปไม่ได้เลย

2. **การประเมินผล** — การตรวจสอบคุณภาพโดยอัตโนมัติช่วยให้แน่ใจว่าการตอบกลับของตัวแทนยังคงถูกต้อง ครบถ้วน และเป็นประโยชน์ตลอดเวลา ตัวแทนประเมินผลสามารถให้คะแนนการตอบกลับตามเกณฑ์ที่กำหนดไว้

3. **การจัดการค่าใช้จ่าย** — การใช้โทเค็นส่งผลโดยตรงต่อค่าใช้จ่าย กลยุทธ์เช่นการเพิ่มประสิทธิภาพคำสั่ง เลือกรูปแบบ และการแคชช่วยควบคุมค่าใช้จ่ายโดยไม่ลดทอนคุณภาพ


## การสร้างเอเจนต์ที่สังเกตได้

เรากำหนดเครื่องมือการเดินทางและห่อหุ้มการเรียกเอเจนต์ด้วยเวลาเพื่อที่เราจะสามารถตรวจสอบความหน่วง ในการใช้งานจริงคุณจะต้องผนวกกับ OpenTelemetry หรือระบบติดตามอื่นที่คล้ายกัน


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evaluation Patterns

A common production pattern is to use a second agent as an **evaluator**. The evaluator scores the primary agent's response against predefined criteria such as completeness, accuracy, and helpfulness.

This enables:
- Automated quality gates before responses reach users
- Regression detection when prompts or models change
- Continuous monitoring of agent performance over time


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## กลยุทธ์การจัดการต้นทุน

การควบคุมต้นทุนเป็นสิ่งสำคัญสำหรับเอเจนต์ AI ที่ผลิต นี่คือกลยุทธ์หลัก:

| กลยุทธ์ | คำอธิบาย |
|---|---|
| **การปรับแต่งพรอมต์** | รักษาคำสั่งของระบบให้ง่าย กระชับ ลบบริบทซ้ำซ้อนเพื่อลดจำนวนโทเค็นอินพุต |
    "| **การเลือกโมเดล** | ใช้โมเดลขนาดเล็กและราคาถูกกว่า (เช่น GPT-4.1-mini) สำหรับงานง่าย ๆ เช่น การจำแนกหรือการสกัด และสงวนโมเดลขนาดใหญ่สำหรับการใช้เหตุผลที่ซับซ้อน |\n",
| **การแคช** | แคชผลลัพธ์เครื่องมือและคำถามที่ใช้งานบ่อยเพื่อหลีกเลี่ยงการเรียก API ซ้ำซ้อน |
| **งบประมาณโทเค็น** | กำหนดขีดจำกัด `max_tokens` เพื่อป้องกันคำตอบที่ยาวเกินคาด |
| **การทำเป็นชุด** | รวมหลายคำถามของผู้ใช้เป็นการเรียก API เดียวเมื่อเป็นไปได้ |

ในทางปฏิบัติ วิธีการแบบหลายชั้นได้ผลดี: ส่งคำขอที่ตรงไปตรงมาถึงโมเดลที่เร็วและราคาถูก และส่งต่อเฉพาะคำถามที่ซับซ้อนไปยังโมเดลที่มีความสามารถมากกว่า


## สรุป

ในบทเรียนนี้คุณได้เรียนรู้วิธีการ:

1. **เพิ่มการสังเกตการณ์** ในการโต้ตอบของเอเย่นต์ด้วยการจับเวลาและการบันทึก เพื่อวางรากฐานสำหรับการติดตามและการตรวจสอบ
2. **ประเมินผลการตอบสนองของเอเย่นต์** โดยอัตโนมัติด้วยเอเย่นต์ประเมินผลที่ให้คะแนนความครบถ้วน ความถูกต้อง และความช่วยเหลือ
3. **จัดการต้นทุน** ผ่านการเพิ่มประสิทธิภาพคำสั่งเลือกโมเดล การแคช และงบประมาณโทเค็น

รูปแบบการผลิตเหล่านี้ช่วยให้มั่นใจว่าเอเย่นต์ AI ของคุณมีความน่าเชื่อถือ วัดผลได้ และคุ้มค่าต้นทุนในระดับใหญ่


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
